In [8]:
try:
    DATASET_NAME = getArgument("DATASET_NAME")
except:
    print("Retail_Observability_Intelligence")

StatementMeta(, 2ee6ef07-c04c-4088-b2e0-3fa3e23c5b55, 10, Finished, Available, Finished, False)

Retail_Observability_Intelligence


In [9]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructType, StructField, IntegerType, DoubleType
from datetime import datetime

SILVER_LAKEHOUSE = "Sales_Intelligence_LH"
DATASET_NAME_CLEAN = DATASET_NAME.lower().replace(' ', '_')
SILVER_TABLE = f"Sales_Intelligence_LH.dbo.silver_{DATASET_NAME_CLEAN}"
RUN_TIMESTAMP      = datetime.now().isoformat()

print(f"Dataset      : {DATASET_NAME}")
print(f"Silver source: {SILVER_TABLE}")
print(f"Run time     : {RUN_TIMESTAMP}")

StatementMeta(, 2ee6ef07-c04c-4088-b2e0-3fa3e23c5b55, 11, Finished, Available, Finished, False)

Dataset      : Retail_Observability_Intelligence
Silver source: Silver_Lakehouse.dbo.silver_retail_observability_intelligence
Run time     : 2026-06-03T07:56:49.756199


In [10]:
from pyspark.sql.types import StringType

silver_df = spark.read.table(SILVER_TABLE)
print(f"Silver rows loaded: {silver_df.count()}")
display(silver_df.limit(5))


# ── Dataset-level severity ────────────────────────────────────
total    = silver_df.count()
flagged  = silver_df.filter(F.col("row_severity") != "HEALTHY").count() \
           if "row_severity" in silver_df.columns else 0
fail_pct = round((flagged / total * 100), 2) if total > 0 else 0.0

if   fail_pct >= 50: run_severity = "CRITICAL"
elif fail_pct >= 25: run_severity = "HIGH"
elif fail_pct > 0:   run_severity = "MEDIUM"
else:                run_severity = "HEALTHY"

print(f"Run severity: {run_severity} ({fail_pct}% flagged)")

# ── Dynamic KPI grouping ──────────────────────────────────────
string_cols = [c for c, t in silver_df.dtypes
               if t == "string"
               and c not in ("row_severity", "source_dataset",
                             "silver_processed_at", "UNKNOWN")]

group_col = "region" if "region" in silver_df.columns \
            else string_cols[0] if string_cols else None

# ── Dynamic numeric aggregation ───────────────────────────────
numeric_cols = [c for c, t in silver_df.dtypes
                if t in ("double", "float", "int", "bigint", "long")
                and c not in ("null_issue_flag", "duplicate_flag",
                              "negative_value_flag", "invalid_region_flag",
                              "row_severity")]

agg_exprs = [F.count("*").alias("total_rows")]

for nc in numeric_cols[:5]:
    agg_exprs.append(F.round(F.sum(nc), 2).alias(f"total_{nc}"))
    agg_exprs.append(F.round(F.avg(nc), 2).alias(f"avg_{nc}"))

for flag in ["null_issue_flag", "duplicate_flag",
             "negative_value_flag", "invalid_region_flag"]:
    if flag in silver_df.columns:
        agg_exprs.append(F.sum(flag).alias(flag.replace("_flag", "_count")))

if "row_severity" in silver_df.columns:
    for sev in ["CRITICAL", "HIGH", "MEDIUM", "HEALTHY"]:
        agg_exprs.append(
            F.count(F.when(F.col("row_severity") == sev, 1))
             .alias(f"{sev.lower()}_rows")
        )

if group_col:
    gold_kpi_df = silver_df.groupBy(group_col).agg(*agg_exprs)
    print(f"Grouped by: [{group_col}]")
else:
    gold_kpi_df = silver_df.agg(*agg_exprs)
    gold_kpi_df = gold_kpi_df.withColumn("group_value", F.lit("ALL"))
    print("No group column — aggregated as single row")

gold_kpi_df = gold_kpi_df \
    .withColumn("dataset_name", F.lit(DATASET_NAME)) \
    .withColumn("run_severity", F.lit(run_severity)) \
    .withColumn("fail_pct",     F.lit(fail_pct)) \
    .withColumn("evaluated_at", F.lit(RUN_TIMESTAMP))

display(gold_kpi_df)

StatementMeta(, 2ee6ef07-c04c-4088-b2e0-3fa3e23c5b55, 12, Finished, Available, Finished, False)

Silver rows loaded: 200


SynapseWidget(Synapse.DataFrame, 7a927c55-7b82-41c6-95fe-a58c6eb9b165)

Run severity: HIGH (31.5% flagged)
Grouped by: [region]


SynapseWidget(Synapse.DataFrame, b3a3cda2-c13f-4c50-ac71-39ef9d1d0dc9)

In [11]:
gold_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Sales_Intelligence_LH.dbo.gold_kpi_by_region")

print(f"✅ Gold KPI written to [{SILVER_LAKEHOUSE}.dbo.gold_kpi_by_region]")

StatementMeta(, 2ee6ef07-c04c-4088-b2e0-3fa3e23c5b55, 13, Finished, Available, Finished, False)

✅ Gold KPI written to [Silver_Lakehouse.dbo.gold_kpi_by_region]


In [12]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

log_schema = StructType([
    StructField("dataset_name",  StringType(),  True),
    StructField("silver_table",  StringType(),  True),
    StructField("total_rows",    IntegerType(), True),
    StructField("flagged_rows",  IntegerType(), True),
    StructField("fail_pct",      DoubleType(),  True),
    StructField("run_severity",  StringType(),  True),
    StructField("evaluated_at",  StringType(),  True),
])

log_df = spark.createDataFrame([(
    DATASET_NAME, SILVER_TABLE,
    int(total), int(flagged),
    float(fail_pct), run_severity, RUN_TIMESTAMP
)], schema=log_schema)

log_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("Sales_Intelligence_LH.dbo.gold_observability_log")

print(f"✅ Observability log written — severity: {run_severity}")

StatementMeta(, 2ee6ef07-c04c-4088-b2e0-3fa3e23c5b55, 14, Finished, Available, Finished, False)

✅ Observability log written — severity: HIGH


In [13]:
kpi = spark.read.table(f"{SILVER_LAKEHOUSE}.dbo.gold_kpi_by_region")
log = spark.read.table(f"{SILVER_LAKEHOUSE}.dbo.gold_observability_log")

print(f"Gold KPI rows : {kpi.count()}")
print(f"Gold log rows : {log.count()}")
display(log.orderBy("evaluated_at", ascending=False))
display(spark.read.table("Sales_Intelligence_LH.dbo.gold_observability_log")
        .orderBy("evaluated_at", ascending=False))

StatementMeta(, 2ee6ef07-c04c-4088-b2e0-3fa3e23c5b55, 15, Finished, Available, Finished, False)

Gold KPI rows : 4
Gold log rows : 9


SynapseWidget(Synapse.DataFrame, 7597ab42-df52-43cd-bc81-7705b70e4ada)